# 03 — Construire le dataset final d'instructions

Ce notebook fusionne les trois partitions dans l'ordre strict demandé : **`bau_bau`**, puis **`fr_bau`**, puis **`fr_fr`**.

Il vérifie les rapports de traduction, aligne les traductions par identifiant et par champ, normalise les marqueurs HTML vides, contrôle l'intégrité du résultat et écrit le dataset final dans `data/03_final/`.

## 1. Configuration et fonctions utilitaires

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "01_splits"
FR_BAU_DIR = PROJECT_ROOT / "data" / "02b_text_browser"
BAU_BAU_DIR = PROJECT_ROOT / "data" / "02c_bau_bau_browser"
FINAL_DIR = PROJECT_ROOT / "data" / "03_final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATASET_PATH = FINAL_DIR / "lree_ai_baule_instruction_55k.jsonl"
FINAL_MANIFEST_PATH = FINAL_DIR / "merge_manifest.json"
ORDER = ("bau_bau", "fr_bau", "fr_fr")
EXPECTED_COUNTS = {"bau_bau": 22_008, "fr_bau": 22_008, "fr_fr": 11_004}

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_atomic(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)

def write_json_atomic(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

BR_PATTERN = re.compile(r"<br\s*/?>", flags=re.IGNORECASE)
NBSP_PATTERN = re.compile(r"&nbsp;", flags=re.IGNORECASE)

def normalize_text(value):
    text = "" if value is None else str(value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = BR_PATTERN.sub("\n", text)
    text = NBSP_PATTERN.sub(" ", text).replace("\u00a0", " ")
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

print("Sortie finale :", FINAL_DATASET_PATH)
print("Ordre         :", " → ".join(ORDER))

## 2. Charger les sources et valider les traductions

In [ ]:
split_manifest = read_json(SPLIT_DIR / "split_manifest.json")
sources = {
    direction: read_jsonl(SPLIT_DIR / f"{direction}_source.jsonl")
    for direction in ORDER
}
translation_reports = {
    "fr_bau": read_json(FR_BAU_DIR / "validation_report.json"),
    "bau_bau": read_json(BAU_BAU_DIR / "validation_report.json"),
}

for direction, expected_count in EXPECTED_COUNTS.items():
    assert len(sources[direction]) == expected_count, (direction, len(sources[direction]), expected_count)
    assert all(row["direction"] == direction for row in sources[direction])

for direction, report in translation_reports.items():
    assert report["completed_batches"] == report["total_batches"], f"Traduction incomplète : {direction}"
    assert report["validation_error_count"] == 0, f"Erreurs de traduction : {direction}"

def translation_index(path, expected_direction):
    records = read_jsonl(path)
    index = {}
    for record in records:
        assert record["direction"] == expected_direction
        assert record["complete"] is True
        assert record["translated_text"] is not None
        key = (record["record_id"], record["target_field"])
        assert key not in index, f"Traduction dupliquée : {key}"
        index[key] = record
    return index

translations = {
    "fr_bau": translation_index(FR_BAU_DIR / "partial_translations.jsonl", "fr_bau"),
    "bau_bau": translation_index(BAU_BAU_DIR / "partial_translations.jsonl", "bau_bau"),
}

expected_fr_bau_keys = {(row["id"], "output_bau") for row in sources["fr_bau"]}
expected_bau_bau_keys = set()
for row in sources["bau_bau"]:
    expected_bau_bau_keys.add((row["id"], "instruction_bau"))
    expected_bau_bau_keys.add((row["id"], "output_bau"))
    if row["input_fr"].strip():
        expected_bau_bau_keys.add((row["id"], "input_bau"))

assert set(translations["fr_bau"]) == expected_fr_bau_keys
assert set(translations["bau_bau"]) == expected_bau_bau_keys

print("Sources chargées      :", {key: len(value) for key, value in sources.items()})
print("Traductions fr_bau   :", len(translations["fr_bau"]))
print("Traductions bau_bau :", len(translations["bau_bau"]))
print("✓ Sources et traductions parfaitement alignées")

## 3. Construire les trois partitions normalisées

In [ ]:
def translated(record_id, target_field, direction):
    return normalize_text(translations[direction][(record_id, target_field)]["translated_text"])

def common_metadata(row, direction, source_language, target_language):
    return {
        "id": row["id"],
        "direction": direction,
        "source_language": source_language,
        "target_language": target_language,
        "source_dataset": row["source_dataset"],
        "source_row_index": row["source_row_index"],
    }

bau_bau_rows = []
for row in sources["bau_bau"]:
    record = common_metadata(row, "bau_bau", "bci", "bci")
    record.update({
        "instruction": translated(row["id"], "instruction_bau", "bau_bau"),
        "input": translated(row["id"], "input_bau", "bau_bau") if row["input_fr"].strip() else "",
        "output": translated(row["id"], "output_bau", "bau_bau"),
    })
    bau_bau_rows.append(record)

fr_bau_rows = []
for row in sources["fr_bau"]:
    record = common_metadata(row, "fr_bau", "fr", "bci")
    record.update({
        "instruction": normalize_text(row["instruction_fr"]),
        "input": normalize_text(row["input_fr"]),
        "output": translated(row["id"], "output_bau", "fr_bau"),
    })
    fr_bau_rows.append(record)

fr_fr_rows = []
for row in sources["fr_fr"]:
    record = common_metadata(row, "fr_fr", "fr", "fr")
    record.update({
        "instruction": normalize_text(row["instruction_fr"]),
        "input": normalize_text(row["input_fr"]),
        "output": normalize_text(row["output_fr"]),
    })
    fr_fr_rows.append(record)

partition_rows = {
    "bau_bau": bau_bau_rows,
    "fr_bau": fr_bau_rows,
    "fr_fr": fr_fr_rows,
}
final_rows = bau_bau_rows + fr_bau_rows + fr_fr_rows
print("Lignes construites :", {key: len(value) for key, value in partition_rows.items()})
print("Total              :", len(final_rows))

## 4. Contrôler l'intégrité et l'ordre final

In [ ]:
expected_total = sum(EXPECTED_COUNTS.values())
direction_counts = Counter(row["direction"] for row in final_rows)
expected_direction_sequence = (
    ["bau_bau"] * EXPECTED_COUNTS["bau_bau"]
    + ["fr_bau"] * EXPECTED_COUNTS["fr_bau"]
    + ["fr_fr"] * EXPECTED_COUNTS["fr_fr"]
)

assert len(final_rows) == expected_total == 55_020
assert len({row["id"] for row in final_rows}) == expected_total
assert [row["direction"] for row in final_rows] == expected_direction_sequence
assert dict(direction_counts) == EXPECTED_COUNTS
assert all(row["instruction"] for row in final_rows)
assert all(row["output"] for row in final_rows)
assert all(row["input"] is not None for row in final_rows)
assert not any(BR_PATTERN.search(row[field]) for row in final_rows for field in ("instruction", "input", "output"))
assert not any(NBSP_PATTERN.search(row[field]) for row in final_rows for field in ("instruction", "input", "output"))

empty_inputs = Counter(row["direction"] for row in final_rows if not row["input"])
print("Répartition       :", dict(direction_counts))
print("Instructions vides :", sum(not row["instruction"] for row in final_rows))
print("Réponses vides     :", sum(not row["output"] for row in final_rows))
print("Inputs vides       :", dict(empty_inputs), "(autorisés)")
print("✓ Intégrité, unicité et ordre final validés")

## 5. Écrire le dataset final et son manifeste

In [ ]:
write_jsonl_atomic(FINAL_DATASET_PATH, final_rows)
final_sha256 = sha256_file(FINAL_DATASET_PATH)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_name": "Lree AI Labs French–Baoulé Instruction Dataset",
    "format": "JSON Lines",
    "schema": ["id", "direction", "source_language", "target_language", "source_dataset", "source_row_index", "instruction", "input", "output"],
    "language_codes": {"fr": "French", "bci": "Baoulé"},
    "ordering": list(ORDER),
    "rows": len(final_rows),
    "partitions": {direction: {"rows": len(partition_rows[direction]), "fraction": len(partition_rows[direction]) / len(final_rows)} for direction in ORDER},
    "empty_inputs": dict(empty_inputs),
    "normalization": {
        "html_break_tags": "converted to newlines; standalone markers become empty inputs",
        "nbsp": "converted to regular spaces",
        "empty_input_allowed": True,
    },
    "translation_validation": translation_reports,
    "source_dataset": split_manifest["source_dataset"],
    "source_license": split_manifest["license"],
    "source_attribution": split_manifest["attribution"],
    "output_path": str(FINAL_DATASET_PATH.relative_to(PROJECT_ROOT)),
    "output_sha256": final_sha256,
}
write_json_atomic(FINAL_MANIFEST_PATH, manifest)

print("Dataset final :", FINAL_DATASET_PATH)
print("Manifeste     :", FINAL_MANIFEST_PATH)
print("SHA-256       :", final_sha256)
print("✓ Dataset final écrit avec succès")

## 6. Vérifier les frontières entre les partitions

In [ ]:
boundary_indices = [
    0,
    EXPECTED_COUNTS["bau_bau"] - 1,
    EXPECTED_COUNTS["bau_bau"],
    EXPECTED_COUNTS["bau_bau"] + EXPECTED_COUNTS["fr_bau"] - 1,
    EXPECTED_COUNTS["bau_bau"] + EXPECTED_COUNTS["fr_bau"],
    len(final_rows) - 1,
]
for index in boundary_indices:
    row = final_rows[index]
    print(f"[{index:05d}] {row['direction']} | {row['id']} | {row['instruction'][:90]!r}")